In [3]:
import os
import torch
import monai
from sklearn.model_selection import train_test_split

def prepare_brats_data(root_dir):
    data_dicts = []
    for group in ['HGG', 'LGG']:
        group_path = os.path.join(root_dir, group)
        if not os.path.exists(group_path): continue
        for patient_id in sorted(os.listdir(group_path)):
            p_dir = os.path.join(group_path, patient_id)
            if not os.path.isdir(p_dir): continue
            
            data_dicts.append({
                "image": [
                    os.path.join(p_dir, f"{patient_id}_t1.nii"),
                    os.path.join(p_dir, f"{patient_id}_t1ce.nii"),
                    os.path.join(p_dir, f"{patient_id}_t2.nii"),
                    os.path.join(p_dir, f"{patient_id}_flair.nii")
                ],
                "label": os.path.join(p_dir, f"{patient_id}_seg.nii")
            })
    return data_dicts

# --- Execute ---
ROOT_DIR = '/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training'
all_files = prepare_brats_data(ROOT_DIR)
train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)
print(f"Total: {len(all_files)} | Train: {len(train_files)} | Val: {len(val_files)}")

Total: 335 | Train: 268 | Val: 67


In [4]:
import monai
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, NormalizeIntensityd, 
    RandSpatialCropd, ToTensord, CropForegroundd, MapTransform
)

class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            result = []
            # TC (Core): 1 + 4 | WT (Whole): 1 + 2 + 4 | ET (Enhancing): 4
            result.append(torch.logical_or(d[key] == 1, d[key] == 4))
            result.append(torch.logical_or(torch.logical_or(d[key] == 1, d[key] == 4), d[key] == 2))
            result.append(d[key] == 4)
            d[key] = torch.stack(result, axis=0).float()
        return d

# Define the Pipeline 
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    
    EnsureChannelFirstd(keys="label", channel_dim="no_channel"),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"), # Convert BEFORE crop
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandSpatialCropd(keys=["image", "label"], roi_size=[128, 128, 128], random_size=False),
    ToTensord(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    EnsureChannelFirstd(keys="label", channel_dim="no_channel"),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ToTensord(keys=["image", "label"]),
])

train_ds = monai.data.Dataset(data=train_files, transform=train_transforms)
train_loader = monai.data.DataLoader(train_ds, batch_size=1, shuffle=True)

val_ds = monai.data.Dataset(data=val_files, transform=val_transforms)
val_loader = monai.data.DataLoader(val_ds, batch_size=1, shuffle=False)

In [5]:
from monai.networks.nets import SegResNet
from monai.losses import DiceFocalLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SegResNet(
    spatial_dims=3, init_filters=32, in_channels=4, out_channels=3
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

loss_function = DiceFocalLoss(smooth_nr=0, smooth_dr=1e-5, squared_pred=True, to_onehot_y=False, sigmoid=True)

In [6]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from monai.inferers import sliding_window_inference

# ================= CONFIG =================

MODEL_PATH  = "/home/23bq1a4293/Personal/project/models/Segmentation/glioma/best_segresnet_brats.pth"
patch_size  = (128, 128, 128)
THRESHOLD   = 0.5
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD MODEL =================

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"✅ Model loaded: {MODEL_PATH}")

# ================= CACHE =================

inference_cache = {}

def run_inference_brats(idx):
    if idx in inference_cache:
        return inference_cache[idx]

    # get sample from val_loader by index
    for i, batch in enumerate(val_loader):
        if i == idx:
            test_batch = batch
            break

    with torch.no_grad():
        out = sliding_window_inference(
            test_batch["image"].to(device),
            roi_size=patch_size,
            sw_batch_size=4,
            predictor=model,
            overlap=0.25,
        )

    img_np   = test_batch["image"].squeeze().cpu().numpy()          # (4, 240, 240, 155)
    label_np = test_batch["label"].squeeze().cpu().numpy()          # (3, 240, 240, 155)
    pred_np  = torch.sigmoid(out).squeeze().cpu().numpy()           # (3, 240, 240, 155)
    pred_bin = (pred_np > THRESHOLD).astype(np.uint8)               # binarized

    # best slice = most whole tumor voxels in GT
    wt_mask   = label_np[1]
    best_slice = int(np.argmax([wt_mask[:, :, i].sum()
                                for i in range(wt_mask.shape[2])]))

    # per-region Dice
    def dice(p, g):
        inter = (p * g).sum()
        denom = p.sum() + g.sum()
        return (2 * inter / denom) if denom > 0 else 0.0

    dice_wt = dice(pred_bin[1], (label_np[1] > 0.5).astype(np.uint8))
    dice_tc = dice(pred_bin[0], (label_np[0] > 0.5).astype(np.uint8))
    dice_et = dice(pred_bin[2], (label_np[2] > 0.5).astype(np.uint8))

    result = {
        "img":        img_np,
        "label":      label_np,
        "pred":       pred_bin,
        "best_slice": best_slice,
        "num_slices": img_np.shape[3],
        "dice_wt":    dice_wt,
        "dice_tc":    dice_tc,
        "dice_et":    dice_et,
        "mean_dice":  float(np.mean([dice_wt, dice_tc, dice_et])),
    }
    inference_cache[idx] = result
    return result

# ================= WIDGETS =================

num_val_patients = len(val_loader)

patient_slider = widgets.IntSlider(
    min=0, max=num_val_patients - 1, step=1, value=0,
    description="Patient:",
    continuous_update=False,
    layout=widgets.Layout(width="400px"),
    style={"description_width": "70px"},
)

slice_slider = widgets.IntSlider(
    min=0, max=154, step=1, value=77,
    description="Z-Slice:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
    style={"description_width": "70px"},
)

modality_toggle = widgets.ToggleButtons(
    options=["FLAIR", "T1ce", "T2"],
    description="Modality:",
    style={"description_width": "70px"},
)

run_button = widgets.Button(
    description="▶  Run Inference",
    button_style="success",
    layout=widgets.Layout(width="160px", height="35px"),
)

best_slice_button = widgets.Button(
    description="⚡ Best Slice",
    button_style="info",
    layout=widgets.Layout(width="130px", height="35px"),
)

dice_label = widgets.HTML(value="<b>Dice:</b> run inference first")
output     = widgets.Output()

# ================= PLOT =================

modality_map = {"FLAIR": 3, "T1ce": 1, "T2": 2}

legend_patches = [
    mpatches.Patch(color='red',    label='Whole Tumor (WT)'),
    mpatches.Patch(color='blue',   label='Tumor Core (TC)'),
    mpatches.Patch(color='purple', label='Enhancing Tumor (ET)'),
]
def make_rgba(mask, color, alpha=0.65):
    """convert binary mask to RGBA with solid color"""
    rgba = np.zeros((*mask.shape, 4))
    c = {
        'red':    (1.0, 0.0, 0.0),
        'blue':   (0.0, 0.4, 1.0),
        'yellow': (1.0, 0.85, 0.0),
    }[color]
    rgba[..., 0] = c[0]
    rgba[..., 1] = c[1]
    rgba[..., 2] = c[2]
    rgba[..., 3] = np.where(mask > 0.5, alpha, 0.0)
    return rgba

def plot_current(idx, slice_idx, modality):
    if idx not in inference_cache:
        with output:
            clear_output(wait=True)
            print(f"⏳ Click '▶ Run Inference' to process patient {idx}")
        return

    r       = inference_cache[idx]
    ch      = modality_map[modality]
    img_sl  = r["img"][ch,   :, :, slice_idx]
    gt_wt   = r["label"][1,  :, :, slice_idx]
    gt_tc   = r["label"][0,  :, :, slice_idx]
    gt_et   = r["label"][2,  :, :, slice_idx]
    pr_wt   = r["pred"][1,   :, :, slice_idx]
    pr_tc   = r["pred"][0,   :, :, slice_idx]
    pr_et   = r["pred"][2,   :, :, slice_idx]

    with output:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.patch.set_facecolor("black")
        for ax in axes.flatten():
            ax.set_facecolor("black")

        # ── Row 1: Modalities ──
        for ax, ch_idx, title in zip(
            axes[0], [3, 1, 2], ["FLAIR (Edema)", "T1ce (Core)", "T2"]
        ):
            ax.imshow(r["img"][ch_idx, :, :, slice_idx], cmap="gray", origin="lower")
            ax.set_title(title, fontsize=11, color="white")
            ax.axis("off")

        # ── Row 2: Overlays ──

        # GT
        axes[1, 0].imshow(img_sl, cmap="gray", origin="lower")
        axes[1, 0].imshow(make_rgba(gt_wt, 'red'),    origin="lower")
        axes[1, 0].imshow(make_rgba(gt_tc, 'blue'),   origin="lower")
        axes[1, 0].imshow(make_rgba(gt_et, 'yellow'), origin="lower")
        axes[1, 0].set_title("Ground Truth", fontsize=11, color="white")
        axes[1, 0].axis("off")

        # Prediction
        axes[1, 1].imshow(img_sl, cmap="gray", origin="lower")
        axes[1, 1].imshow(make_rgba(pr_wt, 'red'),    origin="lower")
        axes[1, 1].imshow(make_rgba(pr_tc, 'blue'),   origin="lower")
        axes[1, 1].imshow(make_rgba(pr_et, 'yellow'), origin="lower")
        axes[1, 1].set_title("Prediction", fontsize=11, color="white")
        axes[1, 1].axis("off")

        # Pred mask only — black background
        axes[1, 2].imshow(make_rgba(pr_wt, 'red'),    origin="lower")
        axes[1, 2].imshow(make_rgba(pr_tc, 'blue'),   origin="lower")
        axes[1, 2].imshow(make_rgba(pr_et, 'yellow'), origin="lower")
        axes[1, 2].set_title("Pred Mask Only", fontsize=11, color="white")
        axes[1, 2].axis("off")

        legend_patches = [
            mpatches.Patch(color='red',                label='Whole Tumor (WT)'),
            mpatches.Patch(color=(0.0, 0.4, 1.0),     label='Tumor Core (TC)'),
            mpatches.Patch(color=(1.0, 0.85, 0.0),    label='Enhancing Tumor (ET)'),
        ]
        fig.legend(handles=legend_patches, loc="lower center",
                   ncol=3, fontsize=10, frameon=True,
                   bbox_to_anchor=(0.5, -0.02), facecolor="white")

        plt.suptitle(
            f"Patient {idx}  |  Slice {slice_idx}  |  {modality}  |  "
            f"Mean Dice: {r['mean_dice']:.4f}  "
            f"WT: {r['dice_wt']:.3f}  TC: {r['dice_tc']:.3f}  ET: {r['dice_et']:.3f}",
            fontsize=12, fontweight="bold", color="white"
        )
        plt.tight_layout()
        plt.show()
# ================= CALLBACKS =================

def on_run_clicked(b):
    idx = patient_slider.value
    run_button.description = "⏳ Running..."
    run_button.disabled    = True
    result = run_inference_brats(idx)
    slice_slider.max   = result["num_slices"] - 1
    slice_slider.value = result["best_slice"]
    dice_label.value   = (
        f"<b>Mean Dice: {result['mean_dice']:.4f}</b> &nbsp;|&nbsp; "
        f"WT: {result['dice_wt']:.3f} &nbsp; "
        f"TC: {result['dice_tc']:.3f} &nbsp; "
        f"ET: {result['dice_et']:.3f}"
    )
    run_button.description = "▶  Run Inference"
    run_button.disabled    = False
    plot_current(idx, slice_slider.value, modality_toggle.value)

def on_best_slice(b):
    idx = patient_slider.value
    if idx in inference_cache:
        slice_slider.value = inference_cache[idx]["best_slice"]

def on_slice_change(change):
    plot_current(patient_slider.value, change["new"], modality_toggle.value)

def on_modality_change(change):
    plot_current(patient_slider.value, slice_slider.value, change["new"])

def on_patient_change(change):
    with output:
        clear_output(wait=True)
        print(f"⏳ Click '▶ Run Inference' to process patient {change['new']}")

run_button.on_click(on_run_clicked)
best_slice_button.on_click(on_best_slice)
slice_slider.observe(on_slice_change, names="value")
modality_toggle.observe(on_modality_change, names="value")
patient_slider.observe(on_patient_change, names="value")

# ================= LAYOUT =================

controls = widgets.VBox([
    widgets.HBox([patient_slider, run_button, best_slice_button]),
    widgets.HBox([slice_slider, modality_toggle]),
    dice_label,
])

display(controls, output)

✅ Model loaded: /home/23bq1a4293/Personal/project/models/Segmentation/glioma/best_segresnet_brats.pth


Output()

In [13]:
%matplotlib inline
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import SimpleITK as sitk
from radiomics import featureextractor

from monai.transforms import (
    Compose, LoadImaged, NormalizeIntensityd, ToTensord,
    Orientationd, EnsureChannelFirstd, MapTransform
)
from monai.networks.nets import SegResNet
from monai.inferers import sliding_window_inference

# ================= CONFIG =================

PATIENT_DIR = "/home/23bq1a4293/Personal/project/seg_data/glioma/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_11_1"
MODEL_PATH  = "/home/23bq1a4293/Personal/project/models/Segmentation/glioma/best_segresnet_brats.pth"
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD MODEL =================

model = SegResNet(
    spatial_dims=3, in_channels=4,
    out_channels=3, init_filters=32
).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"✅ Model loaded: {MODEL_PATH}")

# ================= FILE DISCOVERY =================

files = [f for f in os.listdir(PATIENT_DIR) if f.endswith('.nii') or f.endswith('.nii.gz')]

t1    = os.path.join(PATIENT_DIR, next(f for f in files if "t1"    in f.lower() and "ce" not in f.lower() and "seg" not in f.lower()))
t1ce  = os.path.join(PATIENT_DIR, next(f for f in files if "t1ce"  in f.lower()))
t2    = os.path.join(PATIENT_DIR, next(f for f in files if "t2"    in f.lower() and "flair" not in f.lower() and "seg" not in f.lower()))
flair = os.path.join(PATIENT_DIR, next(f for f in files if "flair" in f.lower()))
seg   = os.path.join(PATIENT_DIR, next(f for f in files if "seg"   in f.lower()))

print(f"  T1    : {os.path.basename(t1)}")
print(f"  T1ce  : {os.path.basename(t1ce)}")
print(f"  T2    : {os.path.basename(t2)}")
print(f"  FLAIR : {os.path.basename(flair)}")
print(f"  SEG   : {os.path.basename(seg)}")

# ================= TRANSFORMS =================

class ConvertToMultiChanneld(MapTransform):
    def __call__(self, data):
        d    = dict(data)
        mask = d["label"].squeeze(0)
        d["label"] = torch.stack([
            torch.logical_or(mask == 1, mask == 4).float(),                              # TC
            torch.logical_or(torch.logical_or(mask == 1, mask == 4), mask == 2).float(), # WT
            (mask == 4).float(),                                                          # ET
        ], dim=0)
        return d

infer_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="label", channel_dim="no_channel"),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ConvertToMultiChanneld(keys="label"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ToTensord(keys=["image", "label"]),
])

# ================= RUN INFERENCE =================

print("\n⏳ Running inference...")
data_dict = {"image": [t1, t1ce, t2, flair], "label": seg}
processed = infer_transforms(data_dict)

input_tensor = processed["image"].unsqueeze(0).to(device)
gt_data      = processed["label"].numpy()   # (3, H, W, D)
img_data     = processed["image"].numpy()   # (4, H, W, D)

with torch.no_grad():
    output   = sliding_window_inference(input_tensor, (128, 128, 128), 4, model, overlap=0.25)
    pred_bin = (torch.sigmoid(output)[0].cpu().numpy() > 0.5).astype(np.uint8)  # (3, H, W, D)

print("✅ Inference complete!")

# ================= DICE SCORES =================

def dice(p, g):
    inter = (p * g).sum()
    denom = p.sum() + g.sum()
    return float(2 * inter / denom) if denom > 0 else 0.0

dice_wt = dice(pred_bin[1], (gt_data[1] > 0.5).astype(np.uint8))
dice_tc = dice(pred_bin[0], (gt_data[0] > 0.5).astype(np.uint8))
dice_et = dice(pred_bin[2], (gt_data[2] > 0.5).astype(np.uint8))
mean_dice = float(np.mean([dice_wt, dice_tc, dice_et]))

print(f"\n📊 Dice Scores:")
print(f"  Whole Tumor (WT) : {dice_wt:.4f}")
print(f"  Tumor Core  (TC) : {dice_tc:.4f}")
print(f"  Enhancing   (ET) : {dice_et:.4f}")
print(f"  Mean Dice        : {mean_dice:.4f}")

# ================= RADIOMICS =================

print("\n⏳ Extracting radiomics features...")

def get_radiomics(image_np, mask_np):
    sitk_img  = sitk.GetImageFromArray(image_np[1])
    sitk_mask = sitk.GetImageFromArray(mask_np[1].astype(np.uint8))
    extractor = featureextractor.RadiomicsFeatureExtractor()
    extractor.enableAllFeatures()
    return extractor.execute(sitk_img, sitk_mask)

radiomics_results = get_radiomics(img_data, pred_bin)

radiomics = {
    "Volume"           : float(radiomics_results.get("original_shape_VoxelVolume", 0)),
    "Sphericity"       : float(radiomics_results.get("original_shape_Sphericity", 0)),
    "Surface Area"     : float(radiomics_results.get("original_shape_SurfaceArea", 0)),
    "Entropy (Texture)": float(radiomics_results.get("original_firstorder_Entropy", 0)),
    "Uniformity"       : float(radiomics_results.get("original_firstorder_Uniformity", 0)),
}

print("\n📊 Radiomics Summary:")
for k, v in radiomics.items():
    print(f"  {k:<20}: {v:.4f}")

# best slice from GT WT
best_slice = int(np.argmax(gt_data[1].sum(axis=(0, 1))))
num_slices = img_data.shape[3]
print(f"\n✅ Best slice: {best_slice}")

# ================= COLOR OVERLAY HELPER =================

def make_rgba(mask, color, alpha=0.65):
    rgba = np.zeros((*mask.shape, 4))
    c = {
        'red':    (1.0, 0.0, 0.0),
        'blue':   (0.0, 0.4, 1.0),
        'yellow': (1.0, 0.85, 0.0),
    }[color]
    rgba[..., 0] = c[0]
    rgba[..., 1] = c[1]
    rgba[..., 2] = c[2]
    rgba[..., 3] = np.where(mask > 0.5, alpha, 0.0)
    return rgba

# ================= WIDGETS =================

slice_slider = widgets.IntSlider(
    min=0, max=num_slices - 1, step=1, value=best_slice,
    description="Z-Slice:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
    style={"description_width": "70px"},
)

modality_toggle = widgets.ToggleButtons(
    options=["FLAIR", "T1ce", "T2"],
    description="Modality:",
    style={"description_width": "70px"},
)

best_slice_btn = widgets.Button(
    description="⚡ Best Slice",
    button_style="info",
    layout=widgets.Layout(width="130px", height="35px"),
)

dice_html = widgets.HTML(
    value=(
        f"<b>Mean Dice: {mean_dice:.4f}</b> &nbsp;|&nbsp; "
        f"WT: {dice_wt:.3f} &nbsp; TC: {dice_tc:.3f} &nbsp; ET: {dice_et:.3f}"
        f"&nbsp;&nbsp;&nbsp;|&nbsp;&nbsp;&nbsp;"
        f"Vol: {radiomics['Volume']:.1f} &nbsp; "
        f"Sphericity: {radiomics['Sphericity']:.3f} &nbsp; "
        f"Entropy: {radiomics['Entropy (Texture)']:.3f}"
    )
)

modality_map = {"FLAIR": 3, "T1ce": 1, "T2": 2}
output       = widgets.Output()

legend_patches = [
    mpatches.Patch(color='red',              label='Whole Tumor (WT)'),
    mpatches.Patch(color=(0.0, 0.4, 1.0),   label='Tumor Core (TC)'),
    mpatches.Patch(color=(1.0, 0.85, 0.0),  label='Enhancing Tumor (ET)'),
]

# ================= PLOT =================

def plot_slice(slice_idx, modality):
    ch     = modality_map[modality]
    img_sl = img_data[ch,    :, :, slice_idx]
    gt_wt  = gt_data[1,      :, :, slice_idx]
    gt_tc  = gt_data[0,      :, :, slice_idx]
    gt_et  = gt_data[2,      :, :, slice_idx]
    pr_wt  = pred_bin[1,     :, :, slice_idx]
    pr_tc  = pred_bin[0,     :, :, slice_idx]
    pr_et  = pred_bin[2,     :, :, slice_idx]

    with output:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.patch.set_facecolor("black")
        for ax in axes.flatten():
            ax.set_facecolor("black")

        # ── Row 1: Modalities ──
        for ax, ch_idx, title in zip(
            axes[0], [3, 1, 2], ["FLAIR (Edema)", "T1ce (Core)", "T2"]
        ):
            ax.imshow(img_data[ch_idx, :, :, slice_idx], cmap="gray", origin="lower")
            ax.set_title(title, fontsize=11, color="white")
            ax.axis("off")

        # ── Row 2 ──

        # GT overlay
        axes[1, 0].imshow(img_sl, cmap="gray", origin="lower")
        axes[1, 0].imshow(make_rgba(gt_wt, 'red'),    origin="lower")
        axes[1, 0].imshow(make_rgba(gt_tc, 'blue'),   origin="lower")
        axes[1, 0].imshow(make_rgba(gt_et, 'yellow'), origin="lower")
        axes[1, 0].set_title("Ground Truth", fontsize=11, color="white")
        axes[1, 0].axis("off")

        # Prediction overlay
        axes[1, 1].imshow(img_sl, cmap="gray", origin="lower")
        axes[1, 1].imshow(make_rgba(pr_wt, 'red'),    origin="lower")
        axes[1, 1].imshow(make_rgba(pr_tc, 'blue'),   origin="lower")
        axes[1, 1].imshow(make_rgba(pr_et, 'yellow'), origin="lower")
        axes[1, 1].set_title("Prediction", fontsize=11, color="white")
        axes[1, 1].axis("off")

        # Pred mask only
        axes[1, 2].imshow(make_rgba(pr_wt, 'red'),    origin="lower")
        axes[1, 2].imshow(make_rgba(pr_tc, 'blue'),   origin="lower")
        axes[1, 2].imshow(make_rgba(pr_et, 'yellow'), origin="lower")
        axes[1, 2].set_title("Pred Mask Only", fontsize=11, color="white")
        axes[1, 2].axis("off")

        fig.legend(handles=legend_patches, loc="lower center",
                   ncol=3, fontsize=10, frameon=True,
                   bbox_to_anchor=(0.5, -0.02), facecolor="white")

        plt.suptitle(
            f"External Patient  |  Slice {slice_idx}  |  {modality}  |  "
            f"Mean Dice: {mean_dice:.4f}  "
            f"WT: {dice_wt:.3f}  TC: {dice_tc:.3f}  ET: {dice_et:.3f}",
            fontsize=12, fontweight="bold", color="white"
        )
        plt.tight_layout()
        plt.show()

# ================= CALLBACKS =================

def on_slice_change(change):
    plot_slice(change["new"], modality_toggle.value)

def on_modality_change(change):
    plot_slice(slice_slider.value, change["new"])

def on_best_slice(b):
    slice_slider.value = best_slice

slice_slider.observe(on_slice_change, names="value")
modality_toggle.observe(on_modality_change, names="value")
best_slice_btn.on_click(on_best_slice)

# ================= LAYOUT =================

controls = widgets.VBox([
    widgets.HBox([slice_slider, best_slice_btn]),
    modality_toggle,
    dice_html,
])

display(controls, output)

# trigger first plot
plot_slice(best_slice, "FLAIR")

✅ Model loaded: /home/23bq1a4293/Personal/project/models/Segmentation/glioma/best_segresnet_brats.pth
  T1    : BraTS19_2013_11_1_t1.nii
  T1ce  : BraTS19_2013_11_1_t1ce.nii
  T2    : BraTS19_2013_11_1_t2.nii
  FLAIR : BraTS19_2013_11_1_flair.nii
  SEG   : BraTS19_2013_11_1_seg.nii

⏳ Running inference...
✅ Inference complete!

📊 Dice Scores:
  Whole Tumor (WT) : 0.5207
  Tumor Core  (TC) : 0.3353
  Enhancing   (ET) : 0.0480
  Mean Dice        : 0.3013

⏳ Extracting radiomics features...


parameter force2D must be set to True to enable shape2D extraction
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated



📊 Radiomics Summary:
  Volume              : 247085.0000
  Sphericity          : 0.4420
  Surface Area        : 43059.6700
  Entropy (Texture)   : 0.9676
  Uniformity          : 0.5223

✅ Best slice: 74


Output()